In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

notebook_path = Path().absolute()
sys.path.append(str(notebook_path.parent))

In [3]:
import torch
from tqdm import tqdm
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from neural_controllers import NeuralController
from utils import newton_dataset

torch.manual_seed(0)
torch.cuda.manual_seed(0)
np.random.seed(0)

/u/skarmakar1/miniconda3/envs/neucon/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
model_type = 'llama'
custom_cache_dir = "/scratch/bbjr/skarmakar/huggingface"

if model_type=='llama':
    model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
    language_model = AutoModelForCausalLM.from_pretrained(
        model_id, device_map="cuda", 
        cache_dir=custom_cache_dir,
    )

    use_fast_tokenizer = "LlamaForCausalLM" not in language_model.config.architectures
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=use_fast_tokenizer, padding_side="left", legacy=False)
    tokenizer.pad_token_id = 0 if tokenizer.pad_token_id is None else tokenizer.pad_token_id
    model_name='llama_3_8b_it'
    
elif model_type=='gemma':

    tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-9b-it")
    language_model = AutoModelForCausalLM.from_pretrained(
        "google/gemma-2-9b-it",
        device_map="auto",
        torch_dtype=torch.bfloat16,
    )
    model_name='gemma_2_9b_it'


Loading checkpoint shards: 100%|██████████| 4/4 [03:16<00:00, 49.17s/it]


In [5]:
controller = NeuralController(
        language_model,
        tokenizer,
        rfm_iters=8,
        batch_size=4,
        control_method='pca'
    )

n_components: 5
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : pca
rfm_iters            : 8
forward_batch_size   : 4
M_batch_size         : 2048
n_components         : 5



In [6]:
concept_types = ["Cam", "Isaac"]
data_dir = "../data/prefixed_newton"

dataset = newton_dataset(data_dir, controller)

Train data: 600
Test data: 324
Train data: 600
Test data: 324


In [7]:
controllers = {}
for concept_type in tqdm(concept_types):
    
    other_type = [k for k in concept_types if k != concept_type][0]
    
    train_data = dataset[concept_type]['train']
    test_data = dataset[concept_type]['test']
        
    controller = NeuralController(
        language_model,
        tokenizer,
        rfm_iters=8,
        batch_size=4,
        # control_method='logistic',
        control_method='rfm',
    )
    
    controller.compute_directions(train_data['inputs'], train_data['labels'])
    
    controllers[concept_type] = controller
     

  0%|          | 0/2 [00:00<?, ?it/s]

n_components: 5
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 4
M_batch_size         : 2048
n_components         : 5

Tuning metric: auc
Getting activations from forward passes


100%|██████████| 120/120 [00:29<00:00,  4.10it/s]


Getting activations from forward passes


100%|██████████| 30/30 [00:07<00:00,  3.87it/s]


...

Optimal M batch size: 480
Time taken for round 0: 0.004175662994384766 seconds
Optimal M batch size: 480
Time taken for round 1: 0.005735635757446289 seconds
Optimal M batch size: 480
Time taken for round 2: 0.00570988655090332 seconds
Optimal M batch size: 480
Time taken for round 3: 0.005721330642700195 seconds
Optimal M batch size: 480
Time taken for round 4: 0.005724191665649414 seconds
Optimal M batch size: 480
Time taken for round 5: 0.0057277679443359375 seconds
Optimal M batch size: 480
Time taken for round 6: 0.0057220458984375 seconds
Optimal M batch size: 480
Time taken for round 7: 0.005733013153076172 seconds
Optimal M batch size: 480
Fitting RFM with ntrain: 480, d: 4096, and nval: 120
Optimal M batch size: 480
Time taken for round 0: 0.0037298202514648438 seconds
Optimal M batch size: 480
Time taken for round 1: 0.005738019943237305 seconds
Optimal M batch size: 480
Time taken for round 2: 0.0057735443115234375 seconds
Optimal M batch size: 480
Time taken for round 3: 0.

Optimal M batch size: 480
Time taken for round 5: 0.005782604217529297 seconds
Optimal M batch size: 480
Time taken for round 6: 0.005742788314819336 seconds
Optimal M batch size: 480
Time taken for round 7: 0.005728721618652344 seconds
Optimal M batch size: 480
Fitting RFM with ntrain: 480, d: 4096, and nval: 120
Optimal M batch size: 480
Time taken for round 0: 0.0037145614624023438 seconds
Optimal M batch size: 480
Time taken for round 1: 0.005746603012084961 seconds
Optimal M batch size: 480
Time taken for round 2: 0.0057220458984375 seconds
Optimal M batch size: 480
Time taken for round 3: 0.005716085433959961 seconds
Optimal M batch size: 480
Time taken for round 4: 0.005730390548706055 seconds
Optimal M batch size: 480
Time taken for round 5: 0.005742549896240234 seconds
Optimal M batch size: 480
Time taken for round 6: 0.0057027339935302734 seconds
Optimal M batch size: 480
Time taken for round 7: 0.005743503570556641 seconds
Optimal M batch size: 480
Best RFM auc: 1.0, reg: 0.

100%|██████████| 31/31 [00:12<00:00,  2.53it/s]


Time taken to compute eigenvectors: 1.273857593536377 seconds


100%|██████████| 2/2 [01:39<00:00, 49.68s/it]


In [18]:
for concept_type in concept_types:
    controller = controllers[concept_type]
    other_type = [k for k in concept_types if k!=concept_type][0]
    
    controller.save(concept=f'{concept_type}', model_name='llama_3_8b_it', path='../directions/')

# Control

In [ ]:
concept_types = ['Cam', 'Isaac']
controllers = {}

for concept_type in concept_types:
    
    controller = NeuralController(
        language_model,
        tokenizer,
        rfm_iters=8,
        control_method='rfm'
        # control_method='logistic'

    )
    
    other_type = [k for k in concept_types if k!=concept_type][0]
    
    controller.load(concept=f'{concept_type}', model_name=model_name, path='../directions/')
    
    controllers[concept_type] = controller
    

In [8]:
# newton_type = "Cam"
newton_type = "Isaac"

raw_inputs = [
    f"What is Cam Newton known for?",
    # f"Why is Newton the phycisist so famous?",
    # f"Why is Isaac Newton so famous?",
    # f"What did Newton contribute to motion?",
]
inputs = [controller.format_prompt(x) for x in raw_inputs]

num_new_tokens = 120
controller = controllers[newton_type]

coef=0.4 #llama 
# coef=9

layers = list(range(-5, -31, -1))
# layers = list(range(-1, -41, -1))

gens=[]
print()
for i in inputs:
    print("Prompt:", i)
    print("===== No Control =====")
    print(controller.generate(i, max_new_tokens=num_new_tokens, do_sample=False).replace(i, ""))
    print()
    
    print(f"===== + {newton_type} Control =====")
    gen = controller.generate(i, layers_to_control=layers, control_coef=coef, 
                                max_new_tokens=num_new_tokens, do_sample=False).replace(i, "")
    gens.append(gen)
    print(gen)
    print()
    print()


Prompt: <|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

What is Cam Newton known for?<|eot_id|>
===== No Control =====


/u/skarmakar1/miniconda3/envs/neucon/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/u/skarmakar1/miniconda3/envs/neucon/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


<|start_header_id|>assistant<|end_header_id|>

Cam Newton is a former American football quarterback who is known for several achievements in his career. Here are some of the key things he is known for:

1. **Heisman Trophy winner**: Cam Newton won the Heisman Trophy in 2010 while playing college football at Auburn University. He was a dominant force in the SEC, leading the Tigers to a national championship that year.

2. **NFL quarterback**: Newton was drafted first overall by the Carolina Panthers in the 2011 NFL Draft. He played for the Panthers from 2011 to 2019 and led the team

===== + Isaac Control =====
<|start_header_id|>assistant<|end_header_id|>

Cam Newton is a renowned English mathematician and physicist. He is known for several significant contributions to the fields of mathematics and science. Some of his notable achievements include:

1. **Development of Calculus**: Newton, along with German mathematician Gottfried Wilhelm Leibniz, is credited with the development of cal